# Surround-grating analysis

This notebook analyzes `spotWithAnnularGrating` with the grating restricted to the
**surround**. It follows the same workflow as the cone center/annulus notebooks:
database refresh, condition discovery, one-cell analysis, persistent saving,
population analysis, and an example-stimulus visualization.

The light level is reconstructed per epoch block. Fixed `EL...` filters come from
the raw Stage device configurator; embedded `FW...` text there is ignored. The
numeric FilterWheel reading comes independently from protected metadata and must
agree across every epoch in a block. The resulting maximum is the calibrated R*/s
at normalized display intensity 1; the actual background is
`max_light_level × backgroundIntensity`.


In [ ]:
import sys
import time

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')

import_started = time.perf_counter()
import numpy as np
import pandas as pd
from IPython.display import display

import retinanalysis as ra
from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import spot_annular_grating as sag

SITE = 'surround'
SITE_LABEL = 'Surround-grating analysis'
MAX_SERIES_RESISTANCE = 30e6
STORE_PATH = sag.store_dir() / f'{SITE}_grating'

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')
print(f'Saved records: {STORE_PATH}')

## 1. Load or refresh the single-cell database

Set `UPDATE_DATABASE = True` when new Symphony H5/JSON metadata should be ingested.
The default connects to the existing local DataJoint database without changing it.


In [ ]:
UPDATE_DATABASE = False
ra.djconnect()

if UPDATE_DATABASE:
    database_report = ra.populate_database()
    print(f"newly added: {len(database_report['added'])}")
    print(f"refreshed: {len(database_report['updated'])}")
    print(f"errored: {len(database_report['skipped'])}")
    print(f"database: {len(database_report['experiments'])} experiments; "
          f"{len(database_report['stale'])} still out of date")
else:
    print('Connected to the existing database. Set UPDATE_DATABASE=True to ingest changes.')


## 2. Search the database for surround-grating recordings

Discovery keeps every recorded cell type, numeric FilterWheel setting, bright-bar
contrast, and bar width. Series resistance still resolves and validates
`onlineAnalysis`. Each row is one explicit cell × mode × NDF combination ×
background × bright contrast × bar-width condition, although the compact table
omits the bright-contrast and database bookkeeping columns requested for this view.

`ndf_combination` and `max_light_level` come from the block-level light reader; no
protocol `maxIntensity` parameter is assumed. `date_index` uses the complete sorted
protocol date list, so the same value works in Section 3 even when a date has no
recording for this grating site.

In [ ]:
protocol_index = sc.find_blocks(sag.PROTOCOL, show=False)
protocol_dates = sorted(protocol_index.exp_name.dropna().unique())
date_index_map = {exp_name: index + 1 for index, exp_name in enumerate(protocol_dates)}

df_blocks = sag.find_blocks(show=False)
df_blocks = sag.check_series_resistance(
    df_blocks, max_series_resistance=MAX_SERIES_RESISTANCE)
site_blocks = df_blocks[df_blocks.grating_site.eq(SITE)].copy()

selected = sag.group_blocks(
    site_blocks, show=False,
    require_filter_wheel=False,
    allowed_bright_contrast=None,
    min_bar_width=None,
    min_epochs=None,
    separate_bright_contrast=True,
    collapse_bar_widths=False)
selected = selected.sort_values(
    ['exp_name', 'cell_label', 'onlineAnalysis', 'ndf_combination',
     'backgroundIntensity', 'bright', 'bar_width']).reset_index(drop=True)
selected.insert(0, 'date_index', selected.exp_name.map(date_index_map).astype(int))

print(f'{SITE_LABEL}: {len(selected)} conditions across '
      f'{selected.exp_name.nunique()} experiments and '
      f"{selected.groupby(['exp_name', 'cell_label']).ngroups} cells")
condition_columns = [
    'date_index', 'exp_name', 'cell_label', 'cell_type_short', 'onlineAnalysis',
    'ndf_combination', 'filter_wheel_ndf', 'max_light_level',
    'backgroundIntensity', 'spot_intensity', 'bar_width', 'aperture',
    'annulus_inner', 'annulus_outer',
]
condition_columns = [column for column in condition_columns if column in selected]
sc.scroll_table(
    selected[condition_columns], height=430,
    num_cols=('date_index', 'filter_wheel_ndf', 'max_light_level',
              'backgroundIntensity', 'spot_intensity', 'bar_width', 'aperture',
              'annulus_inner', 'annulus_outer'))

## 3. Analyze one cell across its recorded conditions

Enter a `date_index`, cell label, and resolved `onlineAnalysis`; no background or
NDF selection is required. This standalone section finds every matching condition
and prints a condition table when there is more than one. Each unique fixed-NDF +
FilterWheel combination, background intensity, and bright-bar contrast is analyzed
separately.

If more than one bright-bar contrast or bar width was recorded, an alert is printed. With
`COLLAPSE_BAR_WIDTHS = False` (the default), each width is also analyzed and saved as
a separate condition. Set it to `True` only when you deliberately want to pool all
bar widths within each otherwise-identical condition.

When multiple fixed-NDF + FilterWheel + background combinations are present,
their tuning curves are overlaid after the individual condition plots. The overlay
labels retain bright-bar contrast and bar width so any additional stimulus changes
remain visible.

Section 3 delegates discovery, analysis, metadata display, and plotting to
`sag.analyze_cell_conditions`, leaving only explicit controls in the notebook.
`DETECTOR_KWARGS` applies center-grating-only detector overrides: candidates from
all epochs in a block are clustered together with one shared polarity and cluster
boundary, plus a minimum peak amplitude of 10. The 5 s trial-section limit remains
configured for the per-epoch fallback; the pooled fit is intentionally not split.
This keeps small neighboring-cell events from becoming the apparent spike cluster in epochs
that contain no large spikes from the recorded cell.
`SPIKE_OFFSET_MS` and `WC_OFFSET_MS` visibly control how far after stimulus
onset each response window starts. Spike counting starts at 0 ms, while the
whole-cell offset remains 100 ms; both windows still stop at `preTime +
stimTime`. Set `SUBTRACT_BASELINE = True` (the default) to subtract the
across-epoch mean preTime response from tuning curves. Set it to `False` to
plot absolute responses instead; the measured baseline is still retained and
drawn as the reference line. Whole-cell traces retain their existing per-epoch
preTime correction.

Before traces are loaded, each condition prints cell identity and time, stimulus
parameters, light calibration, block IDs, and epoch count. Series resistance remains
part of the analysis/QC even though it is omitted from the compact Section 2 table.

In [ ]:
# Standalone after the import cell; Section 2 is optional.
DATE_INDEX = 25
CELL_LABEL = 'Cell1'
ONLINE_ANALYSIS = 'extracellular'  # 'extracellular', 'exc', or 'inh'
COLLAPSE_BAR_WIDTHS = False
SUBTRACT_BASELINE = True
SPIKE_OFFSET_MS = 0.0
WC_OFFSET_MS = 0.0
DETECTOR_KWARGS = {
    'min_peak_amplitude': 10.0,
    'max_trial_length_s': 2.0,
    'cluster_across_trials': True,
    'global_polarity': True,
}

section3 = sag.analyze_cell_conditions(
    date_index=DATE_INDEX,
    cell_label=CELL_LABEL,
    online_analysis=ONLINE_ANALYSIS,
    site=SITE,
    collapse_bar_widths=COLLAPSE_BAR_WIDTHS,
    subtract_baseline=SUBTRACT_BASELINE,
    max_series_resistance=MAX_SERIES_RESISTANCE,
    spike_offset=SPIKE_OFFSET_MS,
    wc_offset=WC_OFFSET_MS,
    detector_kwargs=DETECTOR_KWARGS,
    keep_raw=True,
    plot=True,
    show=True)

# Short aliases retained for Sections 3a and 5.
EXP_NAME = section3.exp_name
condition_rows = section3.condition_rows
light_conditions = section3.light_conditions
records = section3.records
condition_figures = section3.condition_figures
light_tuning_figure = section3.light_tuning_figure


### 3a. Check spike detection on a random subset of epochs

For extracellular recordings, plot raw amplifier traces with detected spikes marked.
This optional check is disabled by default; set `RUN_SPIKE_DETECTION_QC = True`
to run it. `SPIKE_QC_FRACTION` controls the sampled fraction (default 30%), and
`SPIKE_QC_RANDOM_SEED` makes the selection reproducible; set it to `None` to draw a
new random subset each run. Sampling occurs separately within every analyzed light
condition. Use the `QC view` pull-down to inspect one sampled epoch at a time without
stacking every trace vertically. The protocol-independent
`ra.spike_detection_qc_browser` utility handles sampling, deduplication, plotting,
and the compact pull-down display for this and other protocol notebooks.

In [ ]:
RUN_SPIKE_DETECTION_QC = False
SPIKE_QC_FRACTION = 0.30
SPIKE_QC_RANDOM_SEED = 0  # Set to None for a new random subset each run.
SPIKE_QC_EPOCHS_PER_VIEW = 1  # Keep each pull-down view compact.

if RUN_SPIKE_DETECTION_QC and not records:
    raise ValueError('Run Section 3 before checking spike detection')

spike_qc_datasets = []
for condition_index, record in enumerate(
        records if RUN_SPIKE_DETECTION_QC else [], start=1):
    if record.online_analysis != 'extracellular':
        continue
    if record.raw is None or not record.raw.get('traces'):
        print(f'Condition {condition_index}: no raw traces; rerun Section 3 with keep_raw=True.')
        continue

    spike_qc_datasets.append({
        'label': f'Condition {condition_index}',
        'title': (f'Condition {condition_index}: {record.exp_name} | '
                  f'{record.cell_label} | {record.config.get("ndf_combination", "")}'),
        'traces': record.raw['traces'],
        'spike_times': record.raw['spike_times_ms'],
        'sample_rate': record.raw['sample_rate'],
        'spike_time_unit': 'ms',
        'stimulus_window_ms': (
            record.pre_time_ms, record.pre_time_ms + record.stim_time_ms),
        'source_group': record.exp_name,
        'block_ids': record.raw.get('block_id'),
        'epoch_indices': record.raw.get('epoch_index'),
    })

if not RUN_SPIKE_DETECTION_QC:
    print('Spike-detection QC skipped. Set RUN_SPIKE_DETECTION_QC = False to enable it.')
    spike_detection_qc = None
elif not spike_qc_datasets:
    print('No extracellular records are available for spike-detection QC.')
    spike_detection_qc = None
else:
    spike_detection_qc = ra.spike_detection_qc_browser(
        spike_qc_datasets,
        fraction=SPIKE_QC_FRACTION,
        random_state=SPIKE_QC_RANDOM_SEED,
        epochs_per_view=SPIKE_QC_EPOCHS_PER_VIEW)

### 3b. Save these conditions for population analysis

Save every analyzed condition to the site-specific HDF5 store. The key includes
date, cell, mode, site, fixed NDFs, numeric FilterWheel, background, bright-bar
contrast, and the analyzed bar-width set. Therefore multiple light/background/bar
conditions from the same cell remain separate entries; rerunning an identical
condition replaces only that entry.

In [ ]:
if not records:
    raise ValueError('Run Section 3 before saving')
condition_output_path = sag.save_records(records, path=STORE_PATH)
print(f'Saved {len(records)} separate condition(s) to {condition_output_path}')

### 3c. Check saved conditions

Load only the scalar index. Every NDF/background/bright/bar-width condition is one
row; trace arrays remain on disk until a plot requests them.

In [ ]:
saved_cells = sag.load_summary(path=STORE_PATH)
saved_columns = [
    'exp_name', 'cell_label', 'cell_type', 'online_analysis', 'ndf_combination',
    'max_light_level', 'background_intensity', 'bright_bar_contrast',
    'bar_widths', 'rstar', 'series_resistance', 'n_epochs_high_rs',
    'n_epochs', 'block_ids',
]
saved_columns = [column for column in saved_columns if column in saved_cells]
print(f'{len(saved_cells)} saved {SITE}-grating condition(s)')
sc.scroll_table(
    saved_cells[saved_columns], height=320,
    num_cols=('max_light_level', 'background_intensity', 'bright_bar_contrast',
              'rstar', 'series_resistance', 'n_epochs_high_rs', 'n_epochs'))

## 4. Population analysis

This population view uses all extracellular cells recorded with
`brightBarContrast = 0.9`. Missing FilterWheel readings are explicitly assumed to be
FW0 for this analysis and their R* values are recalculated from the recorded fixed
filter stack. The first figure shows every cell's baseline-subtracted tuning curve in
Hz before normalization. The second normalizes each cell before averaging; repeated
records from one cell count only once. Lines show the population mean and shading
shows SEM across cells for the 13,000 R*/s group. Each cell is normalized to its
response at $C_-=-1$; a star marks the cone-model cancellation prediction computed
with $C_+=0.9$ and $W=2000$ R*/s. A third figure shows the same population
after normalizing each cell to its maximum response.

In [ ]:
POPULATION_BRIGHT_CONTRAST = 0.9
POPULATION_MODE = 'extracellular'
POPULATION_LIGHT_BANDS = ((12000.0, 15000.0, 15000.0),)
POPULATION_NORMALIZATION_CONTRAST = -1.0
POPULATION_CONE_W = 2000.0

stored_summary = sag.load_summary(path=STORE_PATH, rstar=False)
summary = sag.add_condition(sag.assume_missing_filter_wheel(
    stored_summary, assumed_ndf=0.0))
if summary.empty:
    raise ValueError(f'No saved {SITE}-grating records in {STORE_PATH}')

patched_fw0 = summary[summary.filter_wheel_assumed].copy()
if not patched_fw0.empty:
    print(f'Assumed FW0 for {len(patched_fw0)} record(s):')
    sc.scroll_table(
        patched_fw0[['exp_name', 'cell_label', 'ndf_combination',
                     'background_intensity', 'rstar']],
        num_cols=('background_intensity', 'rstar'))

population_summary = summary[
    summary.online_analysis.eq(POPULATION_MODE)
    & np.isclose(summary.bright_bar_contrast, POPULATION_BRIGHT_CONTRAST)
].copy()
population_summary = sag.select_population_light_bands(
    population_summary, bands=POPULATION_LIGHT_BANDS)
if population_summary.empty:
    raise ValueError('No records matched the requested population light bands')

population_summary['cell_id'] = (
    population_summary.exp_name.astype(str) + '/'
    + population_summary.cell_label.astype(str))
population_table = (population_summary.groupby('rstar_level', dropna=False)
    .agg(records=('key', 'size'), cells=('cell_id', 'nunique'),
         epochs=('n_epochs', 'sum'), min_rstar=('rstar', 'min'),
         max_rstar=('rstar', 'max'))
    .reset_index())
sc.scroll_table(
    population_table, height=180,
    num_cols=('rstar_level', 'records', 'cells', 'epochs',
              'min_rstar', 'max_rstar'))

population_conditions = tuple(
    population_summary.condition.dropna().drop_duplicates())
population_records = sag.load_records(
    population_summary.key.tolist(), path=STORE_PATH)
cone_prediction_crossing = sag.cone_predict_dark_contrast(
    13000.0, POPULATION_BRIGHT_CONTRAST, POPULATION_CONE_W)
print(f'Cone prediction at 13000 R*/s: C-={cone_prediction_crossing:.4f}')
individual_tuning_figure = sag.plot_population_individual_tuning(
    population_summary, records=population_records, negative_only=True,
    allowed_bright_contrast=(POPULATION_BRIGHT_CONTRAST,),
    figsize=(12.0, 4.8))

population_figure = sag.plot_population_tuning(
    population_summary, records=population_records,
    normalize=True, negative_only=True, min_cells=2,
    normalization_contrast=POPULATION_NORMALIZATION_CONTRAST,
    require_positive_reference=True,
    conditions=population_conditions, modes=(POPULATION_MODE,),
    allowed_bright_contrast=(POPULATION_BRIGHT_CONTRAST,),
    cone_prediction_bright_contrast=POPULATION_BRIGHT_CONTRAST,
    cone_prediction_i0=POPULATION_CONE_W,
    title='Population normalized at C- = -1',
    figsize=(7.2, 5.0))

population_max_response_figure = sag.plot_population_tuning(
    population_summary, records=population_records,
    normalize=True, negative_only=True, min_cells=2,
    normalization_mode='max_response', require_positive_reference=True,
    report_excluded=False,
    conditions=population_conditions, modes=(POPULATION_MODE,),
    allowed_bright_contrast=(POPULATION_BRIGHT_CONTRAST,),
    cone_prediction_bright_contrast=POPULATION_BRIGHT_CONTRAST,
    cone_prediction_i0=POPULATION_CONE_W,
    title='Population normalized to each cell maximum response',
    figsize=(7.2, 5.0))

## 5. Split-field model: net response vs. dark/bright contrast ratio

For an equal-area split field, define the signed ratio $\rho=C_-/C_+$, with
$C_+>0$ and $\beta=I_b/W$. The two half-field responses are combined linearly
*after* each has undergone its intensity-dependent Weber-like gain discount:

$$
R_{\mathrm{net}}(\rho)=\frac{I_bC_+}{2}\left[
\frac{1}{1+\beta(1+C_+)}+
\frac{\rho}{1+\beta(1+\rho C_+)}\right].
$$

The plotted physical domain is $-1/C_+ \leq \rho \leq 0$; its left endpoint
sets the dark-half intensity to zero. Circles mark the analytic cancellation ratio
$\rho_{\mathrm{cancel}}=-(1+\beta)/(1+\beta+2\beta C_+)$, and the dotted
line at $\rho=-1$ marks equal bright and dark contrast magnitudes. Because these
curves represent an OFF cell, the plotted response has the model sign reversed; each
background is divided by its own maximum response so the five curves peak at 1.


In [ ]:
RATIO_RESPONSE_BACKGROUNDS = (1000, 2000, 4000, 8000, 12000)
RATIO_RESPONSE_BRIGHT_CONTRAST = 0.9
RATIO_RESPONSE_W = 2000.0
RATIO_RESPONSE_OFF_CELL = True
RATIO_RESPONSE_NORMALIZE_EACH = True

ratio_response_figure = sag.plot_split_field_ratio_response(
    backgrounds=RATIO_RESPONSE_BACKGROUNDS,
    bright_contrast=RATIO_RESPONSE_BRIGHT_CONTRAST,
    i0=RATIO_RESPONSE_W,
    off_cell=RATIO_RESPONSE_OFF_CELL,
    normalize_each=RATIO_RESPONSE_NORMALIZE_EACH)

## 6. Example surround-grating stimulus

Render the selected block using the recorded aperture, annulus, bar width,
background, spot intensity, and bright/dark contrasts. The geometry should visibly
place the grating over the **surround**; the center spot remains an independent protocol
parameter for surround recordings.


In [ ]:
EXAMPLE_CONDITION_INDEX = 1
if not 1 <= EXAMPLE_CONDITION_INDEX <= len(records):
    raise ValueError(f'EXAMPLE_CONDITION_INDEX must be 1-{len(records)}')
example_record = records[EXAMPLE_CONDITION_INDEX - 1]
example_block = int(example_record.block_ids[0])
stim = ra.StimBlock(example_record.exp_name, example_block, verbose=False)
example_parameters = stim.df_epochs['epoch_parameters'].iloc[0]
dark_values = np.sort(stim.df_epochs['currentDarkContrast'].dropna().unique())
example_dark = dark_values[[0, len(dark_values) // 2, -1]]
stimulus_figure = sag.plot_stimulus_schematic(
    example_parameters, dark_contrasts=example_dark)